# AgriVision — custom disease classifier (MobileNetV2 + PlantVillage)

Free transfer-learning classifier for the pitch: **our own trained model does the classification, Gemini does the reasoning and advice.**
It is *additive*: Gemini Vision stays the primary diagnosis path.

**Before you start**
1. `Runtime` → `Change runtime type` → **T4 GPU** (free tier). This cannot be scripted.
2. Run the cells top to bottom (`Runtime` → `Run all` also works). Expect ~30–45 min total.
3. The last cells upload the model to Firebase Storage (`models/` path) and download the pitch artifacts.

**Honest caveat for the pitch:** PlantVillage photos are single leaves on plain backgrounds. Expect ~98–99% on its test split, but noticeably lower on real field photos. Position it as a proof of our own trained model; say so if asked.

## 1. Setup

In [ ]:
import os
# tf_keras (Keras 2) exports reliably to SavedModel / TFLite / TF.js. Must be set BEFORE importing tensorflow.
os.environ["TF_USE_LEGACY_KERAS"] = "1"
!pip install -q tf_keras scikit-learn matplotlib

In [ ]:
import json, time
import numpy as np
import tensorflow as tf
from tensorflow import keras

print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU found. Runtime > Change runtime type > T4 GPU, then re-run."

## 2. Data — PlantVillage (original curator's GitHub repo)
~54k leaf photos, 38 classes, no login needed. (The TFDS `plant_village` download now fails with HTTP 403, so we use the source repo directly.)
A sparse checkout fetches only the `raw/color` folder (~1 GB); this takes a few minutes. The split is **stratified** 80 / 10 / 10, so every class appears in train, validation and test.

In [ ]:
!rm -rf PlantVillage-Dataset
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git
!cd PlantVillage-Dataset && git sparse-checkout set raw/color
# Fallback if the sparse clone ever fails: !wget -q https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip && unzip -q master.zip

from pathlib import Path
from sklearn.model_selection import train_test_split

root = Path("PlantVillage-Dataset/raw/color")
CLASS_NAMES = sorted(d.name for d in root.iterdir() if d.is_dir())
NUM_CLASSES = len(CLASS_NAMES)
paths, labels = [], []
for idx, cls in enumerate(CLASS_NAMES):
    for f in (root / cls).iterdir():
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
            paths.append(str(f)); labels.append(idx)
paths, labels = np.array(paths), np.array(labels)

p_train, p_hold, y_train, y_hold = train_test_split(paths, labels, test_size=0.2, stratify=labels, random_state=42)
p_val, p_test, y_val, y_test = train_test_split(p_hold, y_hold, test_size=0.5, stratify=y_hold, random_state=42)
print(NUM_CLASSES, "classes |", len(paths), "images | train", len(p_train), "| val", len(p_val), "| test", len(p_test))
assert NUM_CLASSES == 38, f"expected 38 classes, found {NUM_CLASSES}"

In [ ]:
IMG, BATCH, AUTOTUNE = 224, 32, tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.image.resize(img, (IMG, IMG)), label          # float32, 0-255 (scaling to [-1, 1] happens INSIDE the model)

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 20.0)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    return tf.clip_by_value(image, 0.0, 255.0), label

def make(paths, labels, train):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train:
        ds = ds.shuffle(len(paths), seed=42, reshuffle_each_iteration=True)   # shuffles file paths only (cheap)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if train:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make(p_train, y_train, True), make(p_val, y_val, False), make(p_test, y_test, False)

## 3. Model — frozen MobileNetV2 base + new classification head
The model takes raw RGB pixels (float32, 0–255, 224×224) and outputs 38 class probabilities, so the Node inference code needs no model-specific scaling.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input((IMG, IMG, 3), name="image")
x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)   # MobileNetV2 expects [-1, 1]
x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation="softmax", name="probs")(x)
model = keras.Model(inputs, outputs, name="agrivision_disease_classifier")

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train the head (base frozen)

In [ ]:
t0 = time.time()
stop = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[stop])
print(f"head training took {(time.time() - t0) / 60:.1f} min")

## 5. Fine-tune the last layers (low learning rate)

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False                                   # only the last ~30 layers adapt
for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False                               # keep BatchNorm statistics stable

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
t0 = time.time()
stop = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[stop])
print(f"fine-tuning took {(time.time() - t0) / 60:.1f} min")

## 6. Evaluate on the held-out test split

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

os.makedirs("export", exist_ok=True)
y_true = y_test                      # test_ds is not shuffled, so order matches
y_pred = model.predict(test_ds).argmax(axis=1)
accuracy = float((y_true == y_pred).mean())
print(f"TEST ACCURACY: {accuracy:.4f}  ({len(y_true)} images)")

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print(report)
open("export/classification_report.txt", "w").write(f"Test accuracy: {accuracy:.4f}\n\n{report}")

cm = confusion_matrix(y_true, y_pred)
short = [n.replace("___", " – ").replace("_", " ") for n in CLASS_NAMES]
fig, ax = plt.subplots(figsize=(18, 18))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(short, rotation=90, fontsize=7); ax.set_yticklabels(short, fontsize=7)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix — test accuracy {accuracy:.1%}")
fig.colorbar(im, fraction=0.03)
plt.tight_layout(); plt.savefig("export/confusion_matrix.png", dpi=150); plt.show()

## 7. Export — SavedModel, TFLite and TF.js
* **SavedModel** — canonical TensorFlow format.
* **`.tflite`** — small quantized file, for the "runs on-device" pitch line.
* **TF.js layers model** — loads in pure JavaScript, so the Cloud Function needs no native TensorFlow build. Written by hand in the next cell (the `tensorflowjs` Python package is broken on Python 3.13 / current TensorFlow).

In [ ]:
model.save("export/saved_model")                                   # SavedModel

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]               # dynamic-range quantization
open("export/disease_classifier.tflite", "wb").write(converter.convert())
!ls -la export/

In [ ]:
import json, os
import numpy as np

def save_tfjs_layers(model, out_dir, shard_bytes=4 * 1024 * 1024):
    """Write a TF.js layers-model (model.json + weight shards) from a Keras 2 (tf_keras) model."""
    os.makedirs(out_dir, exist_ok=True)
    specs, blobs = [], []
    for w in model.weights:
        arr = np.asarray(w.numpy(), dtype="float32")
        specs.append({"name": w.name.split(":")[0], "shape": list(arr.shape), "dtype": "float32"})
        blobs.append(arr.tobytes())
    names = [s["name"] for s in specs]
    assert len(names) == len(set(names)), "duplicate weight names - TF.js could not match them"
    data = b"".join(blobs)
    n = max(1, -(-len(data) // shard_bytes))
    paths = []
    for i in range(n):
        fname = f"group1-shard{i + 1}of{n}.bin"
        with open(os.path.join(out_dir, fname), "wb") as f:
            f.write(data[i * shard_bytes:(i + 1) * shard_bytes])
        paths.append(fname)
    topology = json.loads(model.to_json())
    doc = {
        "format": "layers-model",
        "generatedBy": "keras v" + str(topology.get("keras_version", "2")),
        "convertedBy": "AgriVision manual exporter",
        "modelTopology": topology,
        "weightsManifest": [{"paths": paths, "weights": specs}],
    }
    with open(os.path.join(out_dir, "model.json"), "w") as f:
        json.dump(doc, f)
    return len(specs), len(data)

n_weights, n_bytes = save_tfjs_layers(model, "export/tfjs_layers")
print(f"TF.js export: {n_weights} weight tensors, {n_bytes / 1e6:.1f} MB")

# Probe: a fixed, reproducible input and Keras's own answer for it, so the exported model can be verified in JavaScript.
i_, j_, k_ = np.meshgrid(np.arange(224), np.arange(224), np.arange(3), indexing="ij")
probe = ((i_ * 7 + j_ * 13 + k_ * 29) % 256).astype("float32")[None]
probs = model.predict(probe, verbose=0)[0]
json.dump({"formula": "(i*7 + j*13 + k*29) % 256, shape [1,224,224,3]", "output": [float(x) for x in probs]}, open("export/probe_output.json", "w"))
print("probe top class:", CLASS_NAMES[int(np.argmax(probs))], "| max prob", round(float(probs.max()), 4))

meta = {
    "model": "disease-classifier-v1",
    "architecture": "MobileNetV2 (ImageNet) + GAP + Dropout + Dense(softmax)",
    "input": {"shape": [1, IMG, IMG, 3], "dtype": "float32", "range": "0-255 RGB (scaling is inside the model)"},
    "dataset": "PlantVillage (spMohanty/PlantVillage-Dataset raw/color), stratified 80/10/10 split",
    "test_accuracy": accuracy,
    "test_images": int(len(y_true)),
    "labels": CLASS_NAMES,
}
json.dump(meta, open("export/labels.json", "w"), indent=2)
json.dump({"test_accuracy": accuracy, "test_images": int(len(y_true))}, open("export/metrics.json", "w"), indent=2)

!du -sh export/saved_model export/disease_classifier.tflite export/tfjs_layers

## 8. Upload to Firebase Storage (`models/` path)
Signs in with your own Google account (the owner of `agrivision-768d1`). The files go under `models/`, **not** `uploads/`, so they stay outside the public upload flow — `storage.rules` only exposes `uploads/`, so clients cannot read `models/`; only the Cloud Function (admin) can.

In [ ]:
from google.colab import auth
auth.authenticate_user()

BUCKET = "gs://agrivision-768d1.firebasestorage.app"
DEST = f"{BUCKET}/models/disease-classifier-v1"
!gcloud config set project agrivision-768d1
!gsutil -m cp -r export/tfjs_layers export/disease_classifier.tflite export/labels.json export/metrics.json {DEST}/
!cd export && zip -qr saved_model.zip saved_model && gsutil cp saved_model.zip {DEST}/
!gsutil ls -r {DEST}

## 9. Download the pitch-deck artifacts

In [ ]:
from google.colab import files
for f in ["export/confusion_matrix.png", "export/classification_report.txt", "export/metrics.json"]:
    files.download(f)

### Done — send back
Tell Claude the **test accuracy** printed in section 6 and whether the `gsutil ls` output in section 8 lists `models/disease-classifier-v1/`. The next step (wiring the classifier into the Cloud Function) needs the package decision in the chat.

## 10. (Optional) Download the model for local testing
Lets Claude test the real model on your machine without depending on Gemini quota. Run after section 7 has finished.


In [ ]:
!cd export && zip -qr tfjs_model.zip tfjs_layers labels.json probe_output.json
from google.colab import files
files.download("export/tfjs_model.zip")
